In [1]:
import pandas as pd
align_sam = pd.read_csv("bowtie_align_mico_103569.sam", sep='\t', header=None, skiprows=11614)
columns = ['read_name', 'flag_sum', 'ref_seq_name', 'fwd_offset', 'map_qual', 'cigar', 'ref_seq_mate_name', 'fwd_offset_mate', 'fragment_len', 'read_seq', 'ASCII', 'AS', 'XS', 'XN', 'XM', 'XO', 'XG', 'NM', 'MD', 'YT']
align_sam.columns = columns
align_sam.head()

,read_name,flag_sum,ref_seq_name,fwd_offset,map_qual,cigar,ref_seq_mate_name,fwd_offset_mate,fragment_len,read_seq,ASCII,AS,XS,XN,XM,XO,XG,NM,MD,YT
0,CRCBB37399-23|TX0228,0,CRCBB37399-23|TX0228,1,6,312M,*,0,0,CCTATCTTCCGTAATCGCACATACTGGTTCAGCTGTAGATTTCTCA...,IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII...,AS:i:0,XS:i:-6,XN:i:0,XM:i:0,XO:i:0,XG:i:0,NM:i:0,MD:Z:312,YT:Z:UU
1,CRCBB37399-23|TX0228,256,CRCBB6068-23|TX0228,1,255,312M,*,0,0,CCTATCTTCCGTAATCGCACATACTGGTTCAGCTGTAGATTTCTCA...,IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII...,AS:i:-6,XS:i:-6,XN:i:0,XM:i:1,XO:i:0,XG:i:0,NM:i:1,MD:Z:311A0,YT:Z:UU
2,CRCBB37399-23|TX0228,256,CRCBB5957-23|TX0228,1,255,306M1I5M,*,0,0,CCTATCTTCCGTAATCGCACATACTGGTTCAGCTGTAGATTTCTCA...,IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII...,AS:i:-20,XS:i:-6,XN:i:0,XM:i:2,XO:i:1,XG:i:1,NM:i:3,MD:Z:308T0A1,YT:Z:UU
3,CRCBB37399-23|TX0228,256,CRCBA44985-23|TX0228,1,255,306M2I4M,*,0,0,CCTATCTTCCGTAATCGCACATACTGGTTCAGCTGTAGATTTCTCA...,IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII...,AS:i:-23,XS:i:-6,XN:i:0,XM:i:2,XO:i:1,XG:i:2,NM:i:4,MD:Z:307T1A0,YT:Z:UU
4,CRCBB37399-23|TX0228,256,CRCBB13830-23|TX0228,1,255,306M1I5M,*,0,0,CCTATCTTCCGTAATCGCACATACTGGTTCAGCTGTAGATTTCTCA...,IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII...,AS:i:-44,XS:i:-6,XN:i:0,XM:i:6,XO:i:1,XG:i:1,NM:i:7,MD:Z:99T80A44C53G28T0A1,YT:Z:UU


In [7]:
read_name = set(align_sam['read_name'])
ref_seq_name = set(align_sam['ref_seq_name'])
intersection = len(read_name.intersection(ref_seq_name))
print(f"Number of unique read names: {len(read_name)}, Number of unique reference sequence names: {len(ref_seq_name)}, Intersection: {intersection}")

Number of unique read names: 11612, Number of unique reference sequence names: 11612, Intersection: 11612


In [ ]:
query = align_sam['']

In [13]:
simple_sam = align_sam[['read_name', 'ref_seq_name', 'NM']].copy()
simple_sam[['seq_id', 'tax_id']] = simple_sam['read_name'].str.split('|', expand=True)
simple_sam[['ref_seq_id', 'ref_tax_id']] = simple_sam['ref_seq_name'].str.split('|', expand=True)
simple_sam = simple_sam[['seq_id', 'tax_id', 'ref_seq_id', 'ref_tax_id', 'NM']]
simple_sam['NM'] = simple_sam['NM'].str.split(':').str[2]
simple_sam = simple_sam.rename(columns={'NM': 'edit_distance'})
simple_sam

,seq_id,tax_id,ref_seq_id,ref_tax_id,edit_distance
0,CRCBB37399-23,TX0228,CRCBB37399-23,TX0228,0
1,CRCBB37399-23,TX0228,CRCBB6068-23,TX0228,1
2,CRCBB37399-23,TX0228,CRCBB5957-23,TX0228,3
3,CRCBB37399-23,TX0228,CRCBA44985-23,TX0228,4
4,CRCBB37399-23,TX0228,CRCBB13830-23,TX0228,7
...,...,...,...,...,...
231273,TSMTA25208-22,TX0228,CRATA40674-22,TX0228,31
231274,TSMTA25208-22,TX0228,EMCEB2400-22,TX0228,31
231275,TSMTA25208-22,TX0228,CRALA17155-21,TX0228,32
231276,TSMTA25208-22,TX0228,CRHTA13812-22,TX0228,31


In [14]:
tax_mapping = pd.read_csv("../total_input-id-map.tsv", sep='\t', header=None, names=['tax_id', 'tax_name'])
id_to_name = dict(zip(tax_mapping['tax_id'], tax_mapping['tax_name']))

simple_sam['tax_name'] = simple_sam['tax_id'].map(id_to_name)
simple_sam['ref_tax_name'] = simple_sam['ref_tax_id'].map(id_to_name)

simple_sam = simple_sam[['seq_id', 'tax_id', 'tax_name', 'ref_seq_id', 'ref_tax_id', 'ref_tax_name', 'edit_distance']]

simple_sam.head()

,seq_id,tax_id,tax_name,ref_seq_id,ref_tax_id,ref_tax_name,edit_distance
0,CRCBB37399-23,TX0228,Cecidomyiidae|Animalia|Arthropoda|Insecta|Dipt...,CRCBB37399-23,TX0228,Cecidomyiidae|Animalia|Arthropoda|Insecta|Dipt...,0
1,CRCBB37399-23,TX0228,Cecidomyiidae|Animalia|Arthropoda|Insecta|Dipt...,CRCBB6068-23,TX0228,Cecidomyiidae|Animalia|Arthropoda|Insecta|Dipt...,1
2,CRCBB37399-23,TX0228,Cecidomyiidae|Animalia|Arthropoda|Insecta|Dipt...,CRCBB5957-23,TX0228,Cecidomyiidae|Animalia|Arthropoda|Insecta|Dipt...,3
3,CRCBB37399-23,TX0228,Cecidomyiidae|Animalia|Arthropoda|Insecta|Dipt...,CRCBA44985-23,TX0228,Cecidomyiidae|Animalia|Arthropoda|Insecta|Dipt...,4
4,CRCBB37399-23,TX0228,Cecidomyiidae|Animalia|Arthropoda|Insecta|Dipt...,CRCBB13830-23,TX0228,Cecidomyiidae|Animalia|Arthropoda|Insecta|Dipt...,7


In [15]:
# split tax_name and ref_tax_name by '|'
simple_sam = simple_sam.copy()
simple_sam['seq_family'] = simple_sam['tax_name'].str.split('|').str[5]
simple_sam['seq_genus'] = simple_sam['tax_name'].str.split('|').str[6]
simple_sam['seq_species'] = simple_sam['tax_name'].str.split('|').str[7]
simple_sam['ref_seq_family'] = simple_sam['ref_tax_name'].str.split('|').str[5]
simple_sam['ref_seq_genus'] = simple_sam['ref_tax_name'].str.split('|').str[6]
simple_sam['ref_seq_species'] = simple_sam['ref_tax_name'].str.split('|').str[7]
simple_sam = simple_sam[['seq_id', 'tax_id', 'seq_family', 'seq_genus', 'seq_species', 'ref_seq_id', 'ref_tax_id', 'ref_seq_family', 'ref_seq_genus', 'ref_seq_species', 'edit_distance']]
simple_sam.head()

,seq_id,tax_id,seq_family,seq_genus,seq_species,ref_seq_id,ref_tax_id,ref_seq_family,ref_seq_genus,ref_seq_species,edit_distance
0,CRCBB37399-23,TX0228,Cecidomyiidae,nan,nan,CRCBB37399-23,TX0228,Cecidomyiidae,nan,nan,0
1,CRCBB37399-23,TX0228,Cecidomyiidae,nan,nan,CRCBB6068-23,TX0228,Cecidomyiidae,nan,nan,1
2,CRCBB37399-23,TX0228,Cecidomyiidae,nan,nan,CRCBB5957-23,TX0228,Cecidomyiidae,nan,nan,3
3,CRCBB37399-23,TX0228,Cecidomyiidae,nan,nan,CRCBA44985-23,TX0228,Cecidomyiidae,nan,nan,4
4,CRCBB37399-23,TX0228,Cecidomyiidae,nan,nan,CRCBB13830-23,TX0228,Cecidomyiidae,nan,nan,7


In [16]:
simple_sam.to_csv("processed_sam.tsv", sep='\t', index=False)

In [17]:
simple_sam['tax_id'].unique()

array(['TX0228', 'TX0930', 'TX1160', 'TX0269', 'TX1154', 'TX0926',
       'TX0266', 'TX0276', 'TX0275', 'TX0287', 'TX45468', 'TX0232',
       'TX41415', 'TX0290', 'TX13151', 'TX0286', 'TX1207', 'TX0261',
       'TX41095', 'TX1218', 'TX0268', 'TX0923', 'TX1138', 'TX41285',
       'TX0301', 'TX0906', 'TX1442', 'TX38592', 'TX1153'], dtype=object)

In [18]:
simple_sam.shape

(231278, 11)

In [19]:
filtered_sam = simple_sam[simple_sam['tax_id'] != simple_sam['ref_tax_id']]
filtered_sam.shape

(4785, 11)

In [20]:
filtered_sam.to_csv("filtered_processed_sam.tsv", sep='\t', index=False)

In [23]:
align_sam[(align_sam['read_name'] == 'CRCEB14789-22|TX0228') & (align_sam['ref_seq_name'] == 'CRCEC18398-22|TX0266')]

,read_name,flag_sum,ref_seq_name,fwd_offset,map_qual,cigar,ref_seq_mate_name,fwd_offset_mate,fragment_len,read_seq,ASCII,AS,XS,XN,XM,XO,XG,NM,MD,YT
783,CRCEB14789-22|TX0228,256,CRCEC18398-22|TX0266,1,255,310M,*,0,0,TCTATCTTCAGTAATTGCTCATACAGGCTCATCAGTAGATTTTTCT...,IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII...,AS:i:0,XS:i:0,XN:i:0,XM:i:0,XO:i:0,XG:i:0,NM:i:0,MD:Z:310,YT:Z:UU


In [24]:
align_sam[(align_sam['read_name'] == 'CRCEB14789-22|TX0228') & (align_sam['ref_seq_name'] == 'CRCEC18398-22|TX0266')]['read_seq'].values[0]

'TCTATCTTCAGTAATTGCTCATACAGGCTCATCAGTAGATTTTTCTATTTTTTCTTTACATATTGCAGGAATTTCTTCTATTTTAGGCGCGATTAATTTTATTTCAACAATAATAAATATAAAAATTAAATTTTTAAAATTTAATCAAATTTCTTTATTTGTATGATCAATTTTAATTACAACAATTTTATTATTATTATCTTTACCAGTTCTAGCTGGAGCTATTACTATATTATTAACTGACCGAAATTTAAATACTTCCTTTTTTGATCCTATAGGAGGAGGAGATCCAGTTTTATATCAACATTTA'

In [25]:
red_seq = 'TCTATCTTCAGTAATTGCTCATACAGGCTCATCAGTAGATTTTTCTATTTTTTCTTTACATATTGCAGGAATTTCTTCTATTTTAGGCGCGATTAATTTTATTTCAACAATAATAAATATAAAAATTAAATTTTTAAAATTTAATCAAATTTCTTTATTTGTATGATCAATTTTAATTACAACAATTTTATTATTATTATCTTTACCAGTTCTAGCTGGAGCTATTACTATATTATTAACTGACCGAAATTTAAATACTTCCTTTTTTGATCCTATAGGAGGAGGAGATCCAGTTTTATATCAACATTTA'
read_seq_fasta = 'TCTATCTTCAGTAATTGCTCATACAGGCTCATCAGTAGATTTTTCTATTTTTTCTTTACATATTGCAGGAATTTCTTCTATTTTAGGCGCGATTAATTTTATTTCAACAATAATAAATATAAAAATTAAATTTTTAAAATTTAATCAAATTTCTTTATTTGTATGATCAATTTTAATTACAACAATTTTATTATTATTATCTTTACCAGTTCTAGCTGGAGCTATTACTATATTATTAACTGACCGAAATTTAAATACTTCCTTTTTTGATCCTATAGGAGGAGGAGATCCAGTTTTATATCAACATTTA'
red_seq == read_seq_fasta

True

In [ ]:
## Test
import pandas as pd
filtered_sam = pd.read_csv("filtered_processed_sam.tsv", sep='\t')
taxid1 = filtered_sam['seq_id'].unique()
taxid2 = filtered_sam['ref_seq_id'].unfiltered_samique()
cluster_ids = set(taxid1) & set(taxid2)
len(cluster_ids)


1100

In [11]:
total_mico = pd.read_csv("test-mbt/total_primer.tsv", sep='\t', header=None, names=['seq_id', 'primer', 'family', 'seq'])
total_mico.head()
total_mico = total_mico[total_mico['seq_id'].isin(cluster_ids)]
total_mico.to_csv("test-mbt/clustered_mico_90.tsv", sep='\t', index=False, header=False)

In [19]:
test_fams = filtered_sam[(filtered_sam['seq_family'] == 'Hybotidae') & (filtered_sam['ref_seq_family'] == 'Cecidomyiidae')].sort_values('edit_distance')
test_fams

,seq_id,tax_id,seq_family,seq_genus,seq_species,ref_seq_id,ref_tax_id,ref_seq_family,ref_seq_genus,ref_seq_species,edit_distance
20,SSEIB2144-13,TX0930,Hybotidae,NaN,NaN,OPPOC1380-17,TX0228,Cecidomyiidae,NaN,NaN,0
2736,CRALA21821-21,TX0930,Hybotidae,NaN,NaN,CRALA15484-21,TX0228,Cecidomyiidae,NaN,NaN,0
2737,CRALA21821-21,TX0930,Hybotidae,NaN,NaN,CRALA39593-21,TX0228,Cecidomyiidae,NaN,NaN,0
21,SSEIB2144-13,TX0930,Hybotidae,NaN,NaN,GMGSL135-13,TX0228,Cecidomyiidae,NaN,NaN,1
2739,CRALA21821-21,TX0930,Hybotidae,NaN,NaN,CRALB17950-21,TX0228,Cecidomyiidae,NaN,NaN,1
2738,CRALA21821-21,TX0930,Hybotidae,NaN,NaN,CRALA27246-21,TX0228,Cecidomyiidae,NaN,NaN,1
2740,CRALA21821-21,TX0930,Hybotidae,NaN,NaN,CRALB5076-21,TX0228,Cecidomyiidae,NaN,NaN,1
2741,CRALA21821-21,TX0930,Hybotidae,NaN,NaN,CRALB24286-21,TX0228,Cecidomyiidae,NaN,NaN,3
2743,CRALA21821-21,TX0930,Hybotidae,NaN,NaN,CRALB13465-21,TX0228,Cecidomyiidae,NaN,NaN,4
2742,CRALA21821-21,TX0930,Hybotidae,NaN,NaN,CRALB23348-21,TX0228,Cecidomyiidae,NaN,NaN,4


In [20]:
test_fams.to_csv("test-mbt/Hybotidae_cecidomyiidae.tsv", sep='\t', index=False)